<a href="https://colab.research.google.com/github/isabelcordova-hash/Analitica-Empresarial-Integrada/blob/main/LAB_D9_AEI_IC%C3%93RDOVA_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 09 — Analítica de Clientes: Procesos Externos

**Semana 9 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía  
**Dataset real:** Online Retail II — UCI Machine Learning Repository  
**Objetivo:** construir un modelo de churn, caracterizar clientes con RFM, aproximar CLV y seleccionar un umbral de campaña según valor económico.
**Alumna:** Córdova Agurto, Isabel


> El notebook debe terminar en una **decisión comercial cuantificada**, no solo en una métrica de machine learning.

# 1. Pregunta de negocio

**¿A qué clientes debe priorizar la empresa en una campaña de retención y qué umbral de probabilidad maximiza el valor económico esperado?**

### Definición operacional
- Unidad: cliente.
- Ventana de observación: 180 días previos al corte.
- Horizonte: 90 días posteriores.
- `churn = 1`: el cliente no compra durante el horizonte posterior.
- Features: RFM y variables transaccionales calculadas solo con información histórica.

In [ ]:
%pip install -q ucimlrepo scikit-learn plotly openpyxl


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_score, recall_score,
    roc_curve, precision_recall_curve
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
RANDOM_STATE = 42
print("Entorno listo.")

Entorno listo.


# 2. Descarga y trazabilidad del dataset real

In [ ]:
# Importación completa, con alternativa desde el Excel oficial.
try:
    retail = fetch_ucirepo(id=502)
    df = retail.data.original.copy()
except Exception as exc:
    print("Importación API no disponible; se intenta el archivo oficial:", type(exc).__name__)
    from io import BytesIO
    from urllib.request import urlopen
    from zipfile import ZipFile
    url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
    with urlopen(url, timeout=120) as response:
        blob = response.read()
    with ZipFile(BytesIO(blob)) as z:
        name = next(name for name in z.namelist() if name.lower().endswith('.xlsx'))
        sheets = pd.read_excel(BytesIO(z.read(name)), sheet_name=None)
    df = pd.concat(sheets.values(), ignore_index=True)
print("Dataset: Online Retail II | UCI ID: 502")
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())
print("Dimensión:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.head())


Importación API no disponible; se intenta el archivo oficial: DatasetNotFoundError
Dataset: Online Retail II | UCI ID: 502
Ejecución: 2026-10-15T00:29:12.702531+00:00
Dimensión: (1067371, 8)
Columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


**Interpretación:**

La salida confirma la carga exitosa del conjunto de datos "Online Retail II" (UCI ID: 502), el cual está compuesto por 1,067,371 registros transaccionales y 8 variables. Las columnas identificadas (Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID y Country) corresponden exactamente a la estructura esperada para el análisis del comportamiento de compra. Esta validación inicial es un paso necesario para proceder con el control de calidad, ya que permite filtrar en las siguientes etapas las cancelaciones, los valores nulos en el identificador de cliente y las transacciones con cantidades o precios no positivos, garantizando así la integridad de los datos para el posterior cálculo de las métricas RFM y la construcción de la variable objetivo de churn.

## Normalización de nombres de columnas

UCI puede devolver pequeñas variaciones en la escritura de los campos. Esta celda identifica las columnas necesarias sin inventar información.

In [ ]:
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_",""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_","")
        if key in normalized:
            return normalized[key]
    return None

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "CustomerID": find_col(df.columns, ["CustomerID", "Customer ID"]),
    "Country": find_col(df.columns, ["Country"]),
}

print(mapping)

required = ["InvoiceNo","Quantity","InvoiceDate","UnitPrice","CustomerID"]
missing = [k for k in required if mapping[k] is None]
if missing:
    raise ValueError(f"No se identificaron columnas obligatorias: {missing}")

rename_map = {v:k for k,v in mapping.items() if v is not None}
df = df.rename(columns=rename_map)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}


**Interpretación:**

La salida confirma que el mapeo de columnas se ha realizado correctamente, identificando los campos necesarios del dataset (como Invoice, Quantity, InvoiceDate, Price y Customer ID) y estandarizando sus nombres para el análisis posterior. Esta validación es un paso previo esencial que asegura que las siguientes etapas de control de calidad (detección de nulos, cantidades o precios no positivos, y cancelaciones) y el cálculo de las métricas RFM se ejecuten sobre las variables correctas, manteniendo la integridad y trazabilidad de los datos.


# 3. Control de calidad

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

print("Fecha mínima:", df["InvoiceDate"].min())
print("Fecha máxima:", df["InvoiceDate"].max())
print("Nulos CustomerID:", df["CustomerID"].isna().sum())
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())

cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
print("Cancelaciones:", cancel.sum())

Fecha mínima: 2009-12-01 07:45:00
Fecha máxima: 2011-12-09 12:50:00
Nulos CustomerID: 243007
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Cancelaciones: 19494


**Interpretación:**

Los resultados del control de calidad confirman la necesidad de depurar el dataset antes de calcular las métricas RFM y la variable objetivo. El rango temporal (diciembre 2009 a diciembre 2011) ofrece una cobertura histórica suficiente para aplicar la ventana de observación de 180 días y el horizonte de churn de 90 días. Sin embargo, se identifican 243,007 registros sin CustomerID, los cuales deben excluirse porque la unidad de análisis es el cliente y no pueden rastrearse. Asimismo, la presencia de 22,950 cantidades no positivas, 6,207 precios no positivos y 19,494 facturas de cancelación (que inician con 'C') indica transacciones que no representan ventas efectivas. Incluir estos registros distorsionaría el cálculo del valor monetario y la frecuencia, por lo que el filtrado posterior es un paso obligatorio para garantizar la integridad de los datos de entrenamiento del modelo.


In [ ]:
clean = df.loc[
    (~cancel) &
    df["CustomerID"].notna() &
    df["InvoiceDate"].notna() &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

clean["CustomerID"] = clean["CustomerID"].astype(str)
clean["Sales"] = clean["Quantity"] * clean["UnitPrice"]

print("Filas válidas:", len(clean))
print("Clientes válidos:", clean["CustomerID"].nunique())
print("Ventas históricas registradas (£):", round(clean["Sales"].sum(), 2))

Filas válidas: 805549
Clientes válidos: 5878
Ventas históricas registradas (£): 17743429.18


**Interpretación:**

Estos resultados confirman la efectividad del proceso de limpieza de datos. Tras eliminar las transacciones con CustomerID nulo, cantidades o precios no positivos, y facturas de cancelación, se obtuvo una base transaccional limpia de 805,549 registros correspondientes a 5,878 clientes únicos. El volumen total de ventas históricas registradas asciende a £17,743,429.18. Esta depuración es fundamental, ya que garantiza que el universo de clientes a analizar posea un historial de compras trazable y económicamente válido, condición indispensable para el cálculo preciso de las métricas RFM y la construcción robusta de la variable objetivo de churn en las siguientes etapas del modelado.

### Pregunta 1
¿Por qué una cancelación o una transacción sin CustomerID no debe tratarse como compra válida para RFM y churn?

**Respuesta:**

Una cancelación no representa una venta efectiva, por lo que incluiría valores que distorsionarían las métricas de valor monetario y frecuencia. Asimismo, una transacción sin CustomerID no puede vincularse a un cliente específico, lo que imposibilita el cálculo individual de las variables RFM (Recencia, Frecuencia y Monetario) y el seguimiento de su historial para determinar correctamente si ocurrió churn en el horizonte de predicción.

**Intercambio oral:** comente en clase; no se requiere transcripción.

# 4. Construcción temporal del target

In [ ]:
# Corte propuesto
cutoff = pd.Timestamp("2011-09-01")
obs_days = 180
horizon_days = 90

# Ajuste defensivo si la cobertura del dataset fuera distinta
max_date = clean["InvoiceDate"].max()
min_date = clean["InvoiceDate"].min()

if cutoff + pd.Timedelta(days=horizon_days) > max_date:
    cutoff = (max_date - pd.Timedelta(days=horizon_days)).normalize()

obs_start = cutoff - pd.Timedelta(days=obs_days)
target_end = cutoff + pd.Timedelta(days=horizon_days)

if obs_start < min_date:
    raise ValueError("La cobertura temporal no permite construir la ventana histórica solicitada.")

print("Inicio observación:", obs_start)
print("Fecha de corte:", cutoff)
print("Fin horizonte:", target_end)

Inicio observación: 2011-03-05 00:00:00
Fecha de corte: 2011-09-01 00:00:00
Fin horizonte: 2011-11-30 00:00:00


**Interpretación:**

Estos resultados confirman la correcta definición temporal del problema de churn, estableciendo una separación estricta entre el pasado y el futuro para garantizar la ausencia de fuga de información (data leakage). La ventana de observación de 180 días (del 5 de marzo al 1 de septiembre de 2011) se utiliza exclusivamente para calcular las variables predictoras (métricas RFM), mientras que el horizonte de 90 días posteriores al corte (del 1 de septiembre al 30 de noviembre de 2011) se reserva únicamente para etiquetar el comportamiento real del cliente (churn). Esta estructura asegura que el modelo se entrene y evalúe bajo condiciones realistas, simulando un escenario de producción donde solo se dispone de información histórica al momento de tomar la decisión comercial.

In [ ]:
hist = clean[
    (clean["InvoiceDate"] >= obs_start) &
    (clean["InvoiceDate"] < cutoff)
].copy()

future = clean[
    (clean["InvoiceDate"] >= cutoff) &
    (clean["InvoiceDate"] < target_end)
].copy()

print("Transacciones históricas:", len(hist))
print("Transacciones futuras:", len(future))
print("Clientes históricos:", hist["CustomerID"].nunique())
print("Clientes con compra futura:", future["CustomerID"].nunique())

Transacciones históricas: 155362
Transacciones futuras: 151993
Clientes históricos: 2772
Clientes con compra futura: 2839


**Interpretación:**

Estos resultados confirman la correcta división temporal de los datos, garantizando la ausencia de fuga de información (data leakage). Las 155,362 transacciones históricas corresponden exclusivamente a la ventana de observación de 180 días y servirán para calcular las variables predictoras (RFM) de los 2,772 clientes históricos, los cuales conforman la cohorte base para el modelado. Por otro lado, las 151,993 transacciones futuras y los 2,839 clientes con compra futura se utilizan únicamente para etiquetar la variable objetivo (churn), asignando un valor de 0 a los clientes históricos que aparecen en este grupo y 1 a los que no, cumpliendo así con la definición operacional del abandono a 90 días.

# 5. Features RFM y variables transaccionales

In [ ]:
# Resumen por factura para ticket promedio
invoice_level = (
    hist.groupby(["CustomerID","InvoiceNo"], as_index=False)
        .agg(
            InvoiceDate=("InvoiceDate","max"),
            invoice_sales=("Sales","sum")
        )
)

rfm = (
    hist.groupby("CustomerID")
        .agg(
            last_purchase=("InvoiceDate","max"),
            Frequency=("InvoiceNo","nunique"),
            Monetary=("Sales","sum"),
            Products=("StockCode","nunique") if "StockCode" in hist.columns else ("InvoiceNo","size"),
            Lines=("InvoiceNo","size")
        )
        .reset_index()
)

rfm["Recency"] = (cutoff - rfm["last_purchase"]).dt.days

ticket = (
    invoice_level.groupby("CustomerID")
                 .agg(avg_ticket=("invoice_sales","mean"))
                 .reset_index()
)

rfm = rfm.merge(ticket, on="CustomerID", how="left")

future_customers = set(future["CustomerID"].unique())
rfm["churn"] = (~rfm["CustomerID"].isin(future_customers)).astype(int)

display(rfm.head())
print("Clientes modelados:", len(rfm))
print("Tasa de churn:", f"{rfm['churn'].mean():.2%}")

,CustomerID,last_purchase,Frequency,Monetary,Products,Lines,Recency,avg_ticket,churn
0,12347.0,2011-08-02 08:48:00,3,1603.68,50,64,29,534.56,0
1,12348.0,2011-04-05 10:47:00,1,367.00,5,5,148,367.00,0
2,12352.0,2011-03-22 16:08:00,2,280.66,8,15,162,140.33,0
3,12353.0,2011-05-19 17:47:00,1,89.00,4,4,104,89.00,1
4,12354.0,2011-04-21 13:11:00,1,1079.40,58,58,132,1079.40,1


Clientes modelados: 2772
Tasa de churn: 39.29%


**Interpretación:**

La salida confirma la correcta agregación de los datos transaccionales a nivel de cliente dentro de la ventana de observación de 180 días. Se han calculado con éxito las métricas centrales de RFM (Recency, Frequency, Monetary), junto con variables complementarias como Products, Lines y avg_ticket, que enriquecen el perfil predictivo del modelo. La variable objetivo churn se asigna adecuadamente (1 para inactividad en el horizonte de 90 días, 0 en caso contrario), como se evidencia en clientes con alta recencia (ej. 104 y 132 días) etiquetados como 1. Finalmente, la cohorte resultante de 2,772 clientes presenta una tasa de churn del 39.29%, lo que indica un desbalance moderado en la clase objetivo y valida que el filtrado temporal aisló correctamente un conjunto de datos histórico apto para el entrenamiento y la evaluación.

### Pregunta 2
¿Por qué sería leakage utilizar compras posteriores a la fecha de corte para calcular Recency, Frequency o Monetary?

**Respuesta:**

Utilizar compras posteriores a la fecha de corte para calcular las métricas RFM introduce fuga de información (data leakage), ya que el modelo estaría procesando datos del futuro que no estarían disponibles en el momento real de la predicción. Dado que la variable objetivo (churn) se define precisamente por la inactividad del cliente en el periodo posterior al corte, incluir esas transacciones futuras en las variables predictoras crea una correlación artificial y directa con el target. Esto infla de manera engañosa las métricas de rendimiento durante el entrenamiento, haciendo que el modelo parezca altamente preciso, pero provocaría un fracaso total en su implementación real, donde únicamente se dispone de información histórica al momento de tomar la decisión comercial.
**Intercambio oral:** comente en clase; no se requiere transcripción.

# 6. Segmentación RFM

In [ ]:
def safe_qcut(series, q=5, reverse=False):
    if series.nunique() <= 1:
        return pd.Series((q + 1) // 2, index=series.index, dtype=int)
    # Valores empatados reciben el mismo score; grupos no necesariamente iguales.
    percentile = series.rank(method="average", pct=True)
    score = np.ceil(percentile * q).clip(1, q).astype(int)
    return q + 1 - score if reverse else score

rfm["R_score"] = safe_qcut(rfm["Recency"], q=5, reverse=True)
rfm["F_score"] = safe_qcut(rfm["Frequency"], q=5, reverse=False)
rfm["M_score"] = safe_qcut(rfm["Monetary"], q=5, reverse=False)
rfm["RFM_score"] = rfm[["R_score","F_score","M_score"]].sum(axis=1)

def segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champions"
    if row["R_score"] <= 2 and row["F_score"] >= 4:
        return "At_Risk_High_Value"
    if row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Hibernating"
    if row["R_score"] >= 4 and row["F_score"] <= 2:
        return "New_or_Promising"
    return "Regular"

rfm["segment"] = rfm.apply(segment, axis=1)

segment_summary = (
    rfm.groupby("segment")
       .agg(
           clientes=("CustomerID","nunique"),
           churn_rate=("churn","mean"),
           recency_med=("Recency","median"),
           monetary_med=("Monetary","median")
       )
       .sort_values("clientes", ascending=False)
)

display(segment_summary)

,clientes,churn_rate,recency_med,monetary_med
segment,,,,
Regular,1061,0.389255,50.0,534.580
Hibernating,815,0.571779,126.0,257.700
Champions,550,0.094545,13.0,1861.725
New_or_Promising,245,0.530612,20.0,301.480
At_Risk_High_Value,101,0.277228,87.0,895.460


**Interpretación:**

La tabla resume la segmentación RFM de los 2,772 clientes modelados, evidenciando una clara diferenciación en el comportamiento de compra y el riesgo de abandono. El segmento "Champions" agrupa a los clientes más valiosos y leales, presentando la tasa de churn más baja (9.45 %), la mayor mediana monetaria (£1,861.73) y la compra más reciente (13 días). Por el contrario, los segmentos "Hibernating" y "New_or_Promising" muestran las tasas de churn más elevadas (57.18 % y 53.06 %, respectivamente), lo que indica un alto riesgo de pérdida, aunque con un valor monetario mediano significativamente menor. Resulta crítico el segmento "At_Risk_High_Value", que, a pesar de tener una tasa de churn moderada (27.72 %), posee un valor monetario mediano considerable (£895.46), lo que lo convierte en un grupo prioritario para las campañas de retención, dado que su abandono representaría un impacto económico desproporcionado para la empresa.


In [ ]:
fig = px.scatter(
    rfm,
    x="Recency",
    y="Monetary",
    size="Frequency",
    color="segment",
    hover_data=["CustomerID","churn"],
    title="Mapa RFM de clientes"
)
fig.show()

**Interpretación:**

El mapa RFM evidencia una clara diferenciación estructural entre los segmentos de clientes. Los "Champions" (azul) se concentran en el cuadrante inferior izquierdo, presentando baja recencia (0-30 días), alto valor monetario (hasta £120,000) y alta frecuencia (puntos de mayor tamaño), lo que confirma su perfil de clientes activos y valiosos. Los "Hibernating" (rojo) se ubican en el extremo derecho con recencias superiores a 80 días y valores monetarios cercanos a cero, indicando clientes inactivos y de bajo valor. Los "Regular" (verde) ocupan una posición intermedia con recencias entre 40-70 días y monetary moderado. Los "New_or_Promising" (morado) muestran baja recencia pero frecuencia y monetary reducidos, sugiriendo clientes recientes con potencial por desarrollar. Finalmente, los "At_Risk_High_Value" (naranja) presentan recencias elevadas (80-110 días) pero con algunos puntos de monetary significativo, identificando clientes valiosos en proceso de alejamiento que requieren atención prioritaria en la campaña de retención.

# 7. Proxy de CLV para priorización económica

In [ ]:
# Proxy didáctico: ticket medio × frecuencia mensual × 12; proxy de ingresos, no CLV completo
months_obs = obs_days / 30.4375
rfm["freq_monthly"] = rfm["Frequency"] / months_obs
rfm["CLV_proxy_12m"] = rfm["avg_ticket"] * rfm["freq_monthly"] * 12

display(
    rfm[["CustomerID","avg_ticket","Frequency","CLV_proxy_12m","segment","churn"]]
      .sort_values("CLV_proxy_12m", ascending=False)
      .head(10)
)

,CustomerID,avg_ticket,Frequency,CLV_proxy_12m,segment,churn
1100,14646.0,3949.664000,30,240435.796000,Champions,0
2683,18102.0,4777.428421,19,184189.771583,Champions,0
2391,17450.0,3830.664706,17,132141.971250,Champions,0
35,12415.0,6811.491111,9,124394.856417,Champions,0
1227,14911.0,727.320494,81,119544.214667,Champions,0
862,14156.0,2100.314815,27,115070.997917,Champions,0
2622,17949.0,1729.379167,24,84220.765417,Champions,0
1315,15098.0,13305.500000,3,80997.231250,At_Risk_High_Value,1
2412,17511.0,2701.329286,14,76740.262792,Champions,0
1731,16029.0,1062.418387,31,66830.543292,Champions,0


**Interpretación:**

La tabla presenta los 10 clientes con el mayor valor económico proyectado a 12 meses (CLV_proxy_12m) dentro del conjunto de datos analizado. Se observa una clara predominancia del segmento "Champions" (9 de los 10 registros), lo que valida la lógica de la segmentación RFM, ya que estos clientes combinan un alto ticket promedio, alta frecuencia de compra y baja recencia, reflejándose en su baja tasa de abandono (churn = 0).
Un hallazgo crítico es la presencia de un cliente del segmento "At_Risk_High_Value" (ID 15098.0) en esta lista, quien efectivamente presentó churn (churn = 1). Esto subraya la vulnerabilidad comercial de los clientes de alto valor que muestran señales de alejamiento y justifica plenamente su priorización en las campañas de retención. Finalmente, cabe reiterar que este CLV_proxy_12m es una estimación de ingresos brutos extrapolados y no un CLV completo, ya que no incorpora márgenes de ganancia, tasas de descuento ni costos de servicio futuros, pero resulta altamente efectivo para establecer una jerarquía de priorización relativa en la toma de decisiones.

> **Advertencia metodológica:** `CLV_proxy_12m` es una aproximación para el ejercicio de decisión. Es una extrapolación de ingresos históricos a 12 meses. No incorpora margen individual, supervivencia, tasa de descuento ni costos futuros. Se conserva CLV_proxy_12m como nombre operativo. Este horizonte económico difiere de los 90 días usados para churn.

### Pregunta 3
¿Por qué dos clientes con la misma probabilidad de churn podrían recibir decisiones distintas si sus valores económicos son diferentes?

**Respuesta:**

Dos clientes con la misma probabilidad de churn pueden recibir decisiones distintas porque la selección para una campaña de retención se fundamenta en el valor económico esperado de la intervención, no únicamente en el riesgo de abandono. Si un cliente posee un valor económico (proxy de CLV) bajo, el costo de contactarlo y aplicar un incentivo podría superar el margen recuperable, resultando en un valor neto negativo o insignificante. Por el contrario, un cliente con idéntica probabilidad de churn pero un valor económico alto justifica la inversión, ya que el ingreso esperado recuperado supera ampliamente los costos operativos, maximizando así el beneficio neto de la estrategia comercial.


# 8. Modelo de churn con scikit-learn

In [ ]:
# CLV_proxy_12m es proporcional a Monetary: se conserva para la decisión,
# pero no se duplica como predictor del modelo.
features = ["Recency", "Frequency", "Monetary", "Products", "Lines", "avg_ticket"]
model_df = rfm.dropna(subset=features + ["churn"]).copy()
X = model_df[features]
y = model_df["churn"]
idx_train, idx_hold = train_test_split(model_df.index, test_size=0.40,
    random_state=RANDOM_STATE, stratify=y)
idx_val, idx_test = train_test_split(idx_hold, test_size=0.50,
    random_state=RANDOM_STATE, stratify=y.loc[idx_hold])
X_train, y_train = X.loc[idx_train], y.loc[idx_train]
X_val, y_val = X.loc[idx_val], y.loc[idx_val]
X_test, y_test = X.loc[idx_test], y.loc[idx_test]
pipe = Pipeline([("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
pipe.fit(X_train, y_train)
proba_val = pipe.predict_proba(X_val)[:, 1]
proba = pipe.predict_proba(X_test)[:, 1]
print("Clientes entrenamiento / validación / prueba:", len(idx_train), len(idx_val), len(idx_test))
print("ROC-AUC en prueba:", round(roc_auc_score(y_test, proba), 4))
print("Average Precision (AP) en prueba:", round(average_precision_score(y_test, proba), 4))
print("El umbral se selecciona solo en validación. La prueba no se usa para reajustarlo.")


Clientes entrenamiento / validación / prueba: 1663 554 555
ROC-AUC en prueba: 0.7255
Average Precision (AP) en prueba: 0.5926
El umbral se selecciona solo en validación. La prueba no se usa para reajustarlo.


**Interpretación:**

Estos resultados confirman la correcta partición estratificada de los datos (60 % entrenamiento, 20 % validación, 20 % prueba) sobre la cohorte de 2,772 clientes modelados. Un ROC-AUC de 0.7255 indica una capacidad discriminatoria moderada a buena del modelo de regresión logística, lo que significa que existe aproximadamente un 72.55 % de probabilidad de que el modelo asigne una puntuación de riesgo mayor a un cliente que realmente abandonó en comparación con uno que se mantuvo activo. Por su parte, un Average Precision (AP) de 0.5926 es una métrica robusta y preferible a la accuracy en este escenario con desbalance de clases, ya que resume el equilibrio entre precisión y recall en todos los umbrales posibles, enfocándose específicamente en la detección de la clase positiva (churn). Finalmente, la distinción metodológica de seleccionar el umbral de decisión únicamente en el conjunto de validación y reservar la prueba para la evaluación final es crucial, pues garantiza que las métricas reportadas reflejen un rendimiento realista y libre de fuga de información (data leakage), simulando adecuadamente las condiciones de un despliegue operativo.

In [ ]:
# Evaluación inicial con umbral 0.50 SOLO como referencia
pred_050 = (proba >= 0.50).astype(int)

print(classification_report(y_test, pred_050, digits=4))
print("Matriz de confusión:")
print(confusion_matrix(y_test, pred_050))

              precision    recall  f1-score   support

           0     0.7181    0.7181    0.7181       337
           1     0.5642    0.5642    0.5642       218

    accuracy                         0.6577       555
   macro avg     0.6412    0.6412    0.6412       555
weighted avg     0.6577    0.6577    0.6577       555

Matriz de confusión:
[[242  95]
 [ 95 123]]


**Interpretación:**

El reporte de clasificación y la matriz de confusión, evaluados con un umbral por defecto de 0.50, revelan un desempeño moderado del modelo en el conjunto de prueba (accuracy de 65.77 %). Para la clase minoritaria de interés (churn = 1), el modelo logra un recall del 56.42 %, lo que implica que deja de identificar a 95 clientes que realmente abandonaron (falsos negativos), representando una pérdida crítica de oportunidad de retención y de su valor económico futuro. Simultáneamente, la precisión del 56.42 % indica que, de los clientes seleccionados para la campaña, casi la mitad (95 falsos positivos) no tenía intención de abandonar, lo que generaría costos de contacto e incentivo innecesarios.

La simetría en los errores (95 falsos positivos y 95 falsos negativos) bajo este umbral estándar evidencia que la métrica estadística por sí sola no optimiza la decisión comercial. Dado que el costo de un falso negativo (pérdida total del margen recuperable del cliente) es sustancialmente superior al de un falso positivo (solo £5 de contacto), este resultado justifica la necesidad de ajustar el umbral de decisión hacia valores más bajos. Esto permite priorizar la captura de clientes en riesgo (aumentando el recall) y maximizar el valor neto esperado, tal como se demuestra en la evaluación económica posterior del laboratorio.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba)

fig_roc = px.area(
    x=fpr, y=tpr,
    title=f"Curva ROC · AUC={roc_auc_score(y_test, proba):.3f}",
    labels={"x":"False Positive Rate","y":"True Positive Rate"}
)
fig_roc.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line_dash="dash")
fig_roc.show()

fig_pr = px.area(
    x=recall_curve, y=precision_curve,
    title=f"Curva Precision-Recall · AP={average_precision_score(y_test, proba):.3f}",
    labels={"x":"Recall","y":"Precision"}
)
fig_pr.show()

**Interpretación:**

La Curva ROC confirma visualmente la capacidad discriminatoria moderada del modelo (AUC = 0.725), ya que la curva se sitúa consistentemente por encima de la diagonal de azar, indicando que el modelo asigna mayores probabilidades a los clientes que efectivamente abandonaron. Sin embargo, la curva se aplana en su tramo superior, lo que evidencia que para incrementar el True Positive Rate más allá de cierto punto es necesario aceptar un aumento considerable del False Positive Rate, reflejando el trade-off inherente al modelo.

La Curva Precision-Recall complementa este análisis mostrando un comportamiento típico en problemas con desbalance de clases: la precisión parte de un valor alto cuando el recall es muy bajo (el modelo identifica con certeza solo a los casos más evidentes de churn), pero cae rápidamente y se estabiliza alrededor de 0.55-0.60 a medida que se intenta capturar más clientes en riesgo. Este descenso confirma que, al relajar el umbral para aumentar el recall, se incorporan progresivamente más falsos positivos, diluyendo la precisión. El AP de 0.593 resume esta relación, evidenciando que el modelo es útil para priorizar clientes, pero requiere un umbral de decisión cuidadosamente calibrado según el costo económico de cada tipo de error.



### Pregunta 4
¿Qué métrica utilizaría para comparar modelos si churn fuera minoritario: accuracy, ROC-AUC o AP? Justifique según el costo de perder clientes en riesgo.

**Respuesta:**

Utilizaría el Average Precision (AP) o, en su defecto, el ROC-AUC, descartando por completo la accuracy. En un escenario con churn minoritario, la accuracy es una métrica engañosa, ya que un modelo ingenuo que predijera que ningún cliente abandona obtendría un puntaje alto sin detectar ningún caso real. El AP es particularmente adecuado porque resume el equilibrio entre precisión y recall en todos los umbrales posibles, enfocándose específicamente en la clase positiva (churn). Dado que el costo de un falso negativo (no detectar a un cliente que abandonará) implica la pérdida total de su valor económico futuro (CLV), es fundamental priorizar una métrica que evalúe rigurosamente la capacidad del modelo para capturar a estos clientes en riesgo, minimizando a la vez los falsos positivos que elevarían innecesariamente los costos operativos de la campaña de retención.

**Intercambio oral:** comente en clase; no se requiere transcripción.

# 9. Del score al umbral de decisión

In [ ]:
cols = ["CustomerID", "segment", "CLV_proxy_12m", "churn"]
val_result = model_df.loc[idx_val, cols].copy()
val_result["proba_churn"] = proba_val
test_result = model_df.loc[idx_test, cols].copy()
test_result["proba_churn"] = proba
display(val_result.head())


,CustomerID,segment,CLV_proxy_12m,churn,proba_churn
1213,14880.0,Hibernating,474.967042,0,0.557962
1419,15304.0,Regular,534.279583,0,0.277227
1944,16473.0,Regular,199.467083,1,0.552662
1258,14984.0,Hibernating,223.573583,0,0.550544
2394,17455.0,Hibernating,267.241250,1,0.595149


**Interpretación:**

Esta salida corresponde al conjunto de evaluación (validación o prueba), donde se integran las predicciones del modelo (proba_churn) con las características económicas y de segmentación de cada cliente (segment, CLV_proxy_12m) y su etiqueta real (churn).

Se observa que las probabilidades estimadas de abandono oscilan entre 27.7 % y 59.5 %, y no siempre coinciden con el resultado observado; por ejemplo, el cliente 14880.0 presenta una probabilidad alta (55.8 %) pero no abandonó (churn = 0), mientras que el cliente 17455.0 sí abandonó con una probabilidad de 59.5 %. Esta tabla constituye el insumo fundamental para la evaluación económica posterior, ya que permite cruzar el costo de los falsos positivos (contactar innecesariamente a un cliente como el 14880.0) con el valor recuperado de los verdaderos positivos (como el 17455.0). Esto demuestra por qué el umbral de decisión no debe fijarse arbitrariamente en 0.50, sino optimizarse dinámicamente según el valor económico del cliente y los supuestos de costo de la campaña de retención.

## Supuestos económicos del caso

Estos valores **no provienen de UCI**. Son supuestos explícitos para convertir el score en una decisión:

- costo de contacto: **£5** por cliente contactado;
- incentivo: **£15** por verdadero churn que se supone retenido gracias a la intervención;
- efectividad de retención: **25 %**;
- margen recuperable: **30 % del CLV proxy**.

Se evaluarán múltiples umbrales.

El valor es una simulación retrospectiva bajo supuestos, no una ganancia observada ni un efecto causal probado. La evaluación se refiere únicamente al subconjunto indicado. Una campaña real requiere verificar el efecto incremental y validar en otros periodos.


In [ ]:
CONTACT_COST = 5.0
INCENTIVE = 15.0
RETENTION_RATE = 0.25
RECOVERABLE_MARGIN = 0.30

def evaluate_threshold(data, threshold,
                       retention_rate=RETENTION_RATE,
                       contact_cost=CONTACT_COST,
                       incentive=INCENTIVE,
                       recoverable_margin=RECOVERABLE_MARGIN):

    d = data.copy()
    d["contact"] = (d["proba_churn"] >= threshold).astype(int)

    tp_mask = (d["contact"] == 1) & (d["churn"] == 1)

    contacted = int(d["contact"].sum())
    tp = int(tp_mask.sum())
    fp = int(((d["contact"] == 1) & (d["churn"] == 0)).sum())
    fn = int(((d["contact"] == 0) & (d["churn"] == 1)).sum())

    precision = tp / contacted if contacted else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    contact_total = contacted * contact_cost
    expected_retained = tp * retention_rate
    incentive_total = expected_retained * incentive

    recovered_value = (
        d.loc[tp_mask, "CLV_proxy_12m"].sum()
        * retention_rate
        * recoverable_margin
    )

    net_value = recovered_value - contact_total - incentive_total

    return {
        "threshold": threshold,
        "policy": "No contactar" if threshold > 1 else ("Contactar a todos" if threshold == 0 else f"Umbral {threshold:.2f}"),
        "contacted": contacted,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "recovered_value": recovered_value,
        "contact_cost": contact_total,
        "incentive_cost": incentive_total,
        "net_value": net_value
    }

thresholds = np.r_[0.0, np.round(np.arange(0.10, 0.91, 0.05), 2), 1.01]
# 0.0 contacta a todos; 1.01 representa no contactar a nadie.

economic = pd.DataFrame([
    evaluate_threshold(val_result, t)
    for t in thresholds
])

display(economic)

,threshold,policy,contacted,TP,FP,FN,precision,recall,recovered_value,contact_cost,incentive_cost,net_value
0,0.00,Contactar a todos,554,218,336,0,0.393502,1.000000,23488.781591,2770.0,817.50,19901.281591
1,0.10,Umbral 0.10,491,213,278,5,0.433809,0.977064,21584.199163,2455.0,798.75,18330.449163
2,0.15,Umbral 0.15,467,209,258,9,0.447537,0.958716,20471.384378,2335.0,783.75,17352.634378
3,0.20,Umbral 0.20,434,199,235,19,0.458525,0.912844,18362.875266,2170.0,746.25,15446.625266
4,0.25,Umbral 0.25,415,196,219,22,0.472289,0.899083,12010.999706,2075.0,735.00,9200.999706
5,0.30,Umbral 0.30,387,188,199,30,0.485788,0.862385,10985.510109,1935.0,705.00,8345.510109
6,0.35,Umbral 0.35,352,172,180,46,0.488636,0.788991,9208.669303,1760.0,645.00,6803.669303
7,0.40,Umbral 0.40,297,157,140,61,0.528620,0.720183,8157.087159,1485.0,588.75,6083.337159
8,0.45,Umbral 0.45,260,143,117,75,0.550000,0.655963,6794.325713,1300.0,536.25,4958.075713
9,0.50,Umbral 0.50,207,116,91,102,0.560386,0.532110,5570.818872,1035.0,435.00,4100.818872


**Interpretación:**

La tabla de evaluación de umbrales en el conjunto de validación demuestra que la política óptima para maximizar el beneficio económico es "Contactar a todos" (umbral 0.00), la cual genera el mayor valor neto esperado de £19,901.28.

A medida que el umbral de decisión aumenta, se observa el clásico trade-off entre precisión y recall: la precisión mejora (llegando hasta 63.16 % en el umbral 0.65), pero el recall cae drásticamente (de 100 % a 11.01 %). Desde una perspectiva económica, este aumento en la precisión no compensa la pérdida de oportunidad. Al subir el umbral, el modelo deja de identificar a clientes que realmente abandonarían (aumento de falsos negativos), lo que reduce severamente el recovered_value. Dado que el margen recuperado de los verdaderos positivos es sustancialmente mayor que el costo operativo de contactar a los falsos positivos (£5 por contacto), la estrategia más agresiva (umbral 0.00) resulta ser la más rentable, ya que el valor recuperado (£23,488.78) supera con creces la suma de los costos de contacto e incentivo (£3,587.50).

In [ ]:
# Desempate: preferir menos contactos, luego mayor umbral.
def choose_policy(table):
    return table.sort_values(["net_value", "contacted", "threshold"],
                             ascending=[False, True, False]).iloc[0]
best = choose_policy(economic)
final_result = evaluate_threshold(test_result, float(best["threshold"]))
print("MEJOR POLÍTICA ENTRE LAS EVALUADAS EN VALIDACIÓN")
print(best.to_string())
print("EVALUACIÓN EN PRUEBA SIN REAJUSTAR EL UMBRAL")
display(pd.DataFrame([final_result]))


MEJOR POLÍTICA ENTRE LAS EVALUADAS EN VALIDACIÓN
threshold                        0.0
policy             Contactar a todos
contacted                        554
TP                               218
FP                               336
FN                                 0
precision                   0.393502
recall                           1.0
recovered_value         23488.781591
contact_cost                  2770.0
incentive_cost                 817.5
net_value               19901.281591
EVALUACIÓN EN PRUEBA SIN REAJUSTAR EL UMBRAL


,threshold,policy,contacted,TP,FP,FN,precision,recall,recovered_value,contact_cost,incentive_cost,net_value
0,0.0,Contactar a todos,555,218,337,0,0.392793,1.0,15381.579972,2775.0,817.5,11789.079972


**Interpretación:**

Los resultados confirman que la política óptima seleccionada en el conjunto de validación es "Contactar a todos" (umbral 0.00), la cual maximiza el valor neto esperado en £19,901.28 al lograr un recall del 100% (capturando los 218 verdaderos positivos) y un valor recuperado de £23,488.78, superando ampliamente los costos operativos de contacto e incentivo. Al aplicar esta misma política sin reajustarla al conjunto de prueba (555 clientes), se mantiene un recall del 100% y una precisión del 39.28%, generando un valor neto simulado de £11,789.08. Este resultado valida que, bajo los supuestos económicos del caso, el beneficio marginal de retener a todos los clientes en riesgo (verdaderos positivos) compensa el costo de contactar a los falsos positivos, consolidando la estrategia de intervención masiva como la más rentable para este escenario específico.

In [ ]:
fig = px.line(
    economic,
    x="threshold",
    y="net_value",
    markers=True,
    title="Valor neto simulado en validación (1.01 = no contactar)",
    labels={
        "threshold":"Umbral de decisión",
        "net_value":"Valor neto esperado (£)"
    }
)
fig.add_vline(x=float(best["threshold"]), line_dash="dash")
fig.show()

**Interpretación:**

El gráfico evidencia de manera visual que el valor neto esperado disminuye de forma monótona a medida que se incrementa el umbral de decisión, confirmando que la política óptima en validación corresponde al umbral 0.00 ("Contactar a todos"), con un valor neto máximo de aproximadamente £19,901. La caída más pronunciada se observa entre los umbrales 0.20 y 0.25, momento en el que el modelo comienza a dejar de identificar clientes en riesgo (aumento de falsos negativos), lo que reduce drásticamente el valor recuperado. A partir del umbral 0.70, el valor neto se estabiliza en £0, reflejando la política de "no contactar", ya que ningún cliente supera ese umbral de probabilidad. Este comportamiento valida que, bajo los supuestos económicos del caso (costo de contacto de £5 e incentivo de £15 con efectividad del 25%), el beneficio marginal de capturar todos los verdaderos positivos supera ampliamente el costo de los falsos positivos, haciendo que la estrategia más inclusiva sea la más rentable.

In [ ]:
fig = px.line(
    economic,
    x="threshold",
    y=["precision","recall"],
    markers=True,
    title="Trade-off precision–recall según umbral",
    labels={"threshold":"Umbral","value":"Métrica","variable":"Métrica"}
)
fig.show()

**Interpretación:**

El gráfico ilustra el trade-off clásico entre precisión y recall al variar el umbral de decisión. El recall (línea roja) parte de 1.0 con umbral 0.00, capturando el 100% de los clientes que realmente abandonaron, y cae de forma monótona hasta anularse a partir del umbral 0.70. Por el contrario, la precisión (línea azul) inicia en un valor bajo (~0.39) debido a la alta cantidad de falsos positivos al contactar a todos, y aumenta gradualmente hasta alcanzar su máximo (~0.63) en el umbral 0.65, momento en el que el modelo identifica con mayor certeza a los clientes en riesgo, pero a costa de dejar fuera a la mayoría de ellos (recall ~0.11). Esta dinámica confirma que no existe un umbral estadísticamente "perfecto": la elección óptima depende del costo relativo de cada error, siendo en este caso más rentable sacrificar precisión para maximizar el recall y así capturar el valor económico de los clientes en riesgo.

### Pregunta 5
¿La política elegida en validación coincide con el umbral 0.50? Explique la diferencia usando el valor económico.

**Respuesta:**

No, la política elegida en validación no coincide con el umbral de 0.50, sino que corresponde a un umbral de 0.00 ("Contactar a todos"). Esto se debe a que, bajo los supuestos económicos del caso (costo de contacto de £5 e incentivo de £15), el valor recuperado al retener a los clientes que realmente abandonarían supera ampliamente el costo de contactar a los falsos positivos. El umbral de 0.00 maximiza el valor neto esperado en validación (£19,901.28), mientras que el umbral de 0.50 reduce drásticamente este beneficio (£4,100.82) al incrementar los falsos negativos, lo que implica la pérdida total del valor económico futuro de esos clientes.


### Pregunta 6
Al aplicar la política elegida al conjunto de prueba, ¿cuántos clientes se contactarían y cuál es el valor neto esperado bajo los supuestos? No extrapole a toda la cartera.

**Respuesta:**

Al aplicar la política óptima seleccionada en validación (umbral de 0.00) al conjunto de prueba, se contactaría a la totalidad de los clientes evaluados en este subconjunto, es decir, a 555 clientes. Bajo los supuestos económicos definidos, esta decisión genera un valor neto esperado en prueba de £11,789.08, monto que resulta de restar los costos operativos de contacto e incentivos al margen recuperado de los clientes retenidos, sin extrapolar este resultado al resto de la cartera.


# 10. Sensibilidad económica

In [ ]:
scenarios = []
for retention_rate in [0.10, 0.25, 0.40]:
    temp = pd.DataFrame([evaluate_threshold(val_result, t, retention_rate=retention_rate)
                         for t in thresholds])
    selected = choose_policy(temp)
    row = evaluate_threshold(test_result, float(selected["threshold"]), retention_rate=retention_rate)
    row["retention_rate"] = retention_rate
    row["validation_net_value"] = float(selected["net_value"])
    scenarios.append(row)
scenario_df = pd.DataFrame(scenarios)
display(scenario_df[["retention_rate", "policy", "contacted", "precision", "recall", "net_value"]])


,retention_rate,policy,contacted,precision,recall,net_value
0,0.10,Contactar a todos,555,0.392793,1.0,3050.631989
1,0.25,Contactar a todos,555,0.392793,1.0,11789.079972
2,0.40,Contactar a todos,555,0.392793,1.0,20527.527955


**Interpretación:**

La tabla de sensibilidad económica demuestra que, al variar la efectividad de retención (10 %, 25 % y 40 %), la política óptima seleccionada en validación ("Contactar a todos") y sus métricas operativas en prueba (555 clientes contactados, precisión del 39.28 % y recall del 100 %) permanecen inalteradas, ya que las predicciones del modelo son independientes de estos supuestos comerciales. Sin embargo, el valor neto esperado escala proporcionalmente con la efectividad, pasando de £3,050.63 a £20,527.53. Esto enseña que, aunque el rendimiento predictivo del modelo es estático, la decisión comercial final está intrínsecamente condicionada por el contexto económico. El hecho de que la estrategia más agresiva siga siendo rentable incluso en el escenario más conservador (10 %) indica que el margen recuperado de los verdaderos positivos es suficientemente alto como para absorber los costos operativos de los falsos positivos, consolidando la robustez de esta política bajo diferentes condiciones de negocio.

In [ ]:
fig = px.bar(
    scenario_df,
    x="retention_rate",
    y="net_value",
    text="policy",
    title="Valor neto en prueba de cada política seleccionada en validación",
    labels={
        "retention_rate":"Efectividad de retención",
        "net_value":"Valor neto simulado en prueba (£)",
        "threshold":"Umbral óptimo"
    }
)
fig.show()

**Interpretación:**

El gráfico de barras confirma visualmente que, al variar la efectividad de retención entre 10 %, 25 % y 40 %, la política óptima seleccionada en validación se mantiene invariable como "Contactar a todos" (umbral 0.00), mientras que el valor neto simulado en prueba escala de manera directamente proporcional al incremento de la efectividad, pasando de aproximadamente £3,050 a £11,789 y finalmente a £20,527. Esta representación gráfica evidencia que, bajo los supuestos económicos del caso, el margen recuperado de los verdaderos positivos es suficientemente robusto como para justificar la estrategia de intervención masiva incluso en el escenario más conservador (10 %), consolidando la decisión comercial como rentable y robusta frente a la incertidumbre sobre la efectividad real de la campaña de retención.

### Pregunta 7
Al comparar efectividades de 10 %, 25 % y 40 %, ¿cambian la política elegida, los clientes contactados y el valor neto en prueba? Explique qué enseña esto sobre predicción y decisión.

**Respuesta:**

Al comparar las efectividades de retención del 10 %, 25 % y 40 %, la política elegida ("Contactar a todos", umbral 0.00) y la cantidad de clientes contactados en el conjunto de prueba (555) no cambian. Únicamente varía el valor neto esperado, el cual aumenta proporcionalmente con la efectividad (de £3,050.63 a £20,527.53). Esto enseña que, si bien la predicción del modelo (sus probabilidades, precisión y recall) permanece constante e independiente de los supuestos comerciales, la decisión óptima depende de la interacción entre el rendimiento del modelo y el contexto económico. En este caso específico, el valor recuperado de los verdaderos positivos es lo suficientemente alto como para absorber el costo de los falsos positivos incluso en el escenario más conservador (10 %), manteniendo la política más agresiva como la óptima. Sin embargo, demuestra que la decisión comercial no se deriva automáticamente de la métrica estadística, sino que está condicionada a que el beneficio financiero supere los costos operativos bajo los parámetros del negocio.


# 11. Priorización por segmento RFM

In [ ]:
segment_econ = (
    test_result.groupby("segment")
               .agg(
                   clientes=("CustomerID","nunique"),
                   churn_rate=("churn","mean"),
                   proba_media=("proba_churn","mean"),
                   clv_medio=("CLV_proxy_12m","mean"),
                   clv_total=("CLV_proxy_12m","sum")
               )
               .sort_values("clv_total", ascending=False)
)

display(segment_econ)

,clientes,churn_rate,proba_media,clv_medio,clv_total
segment,,,,,
Champions,113,0.115044,0.095179,9163.556129,1.035482e+06
Regular,218,0.389908,0.369099,1557.564598,3.395491e+05
Hibernating,161,0.577640,0.600383,782.900771,1.260470e+05
At_Risk_High_Value,20,0.100000,0.264459,2161.276577,4.322553e+04
New_or_Promising,43,0.581395,0.472799,756.134557,3.251379e+04


**Interpretación:**

La tabla valida la estratificación económica y comportamental de los segmentos en el conjunto de prueba. El segmento "Champions" concentra la mayor parte del valor económico total (£1.03 millones) con el menor riesgo de abandono (11.5 %), consolidándose como la base más rentable y leal de la cartera. Por el contrario, "Hibernating" y "New_or_Promising" presentan las tasas de churn más elevadas (superiores al 57 %) y el menor valor monetario medio, lo que los hace económicamente ineficientes para campañas de retención costosas. Finalmente, el segmento "At_Risk_High_Value", aunque registra una tasa de churn observada baja (10 %) en esta muestra, mantiene un valor monetario medio alto (£2,161), lo que justifica su priorización estratégica, ya que la pérdida de estos pocos clientes representaría un impacto desproporcionado en los ingresos.


### Preguntas 8 y 9 — Integrar en el cierre ejecutivo
8. ¿Qué segmento priorizaría o descartaría? Sustente con el riesgo y el valor económico observados.
9. Indique una limitación y una condición que haría cambiar la decisión.

**Respuesta:**

El churn se define operacionalmente como la inactividad de 90 días posteriores a la fecha de corte (01-sep-2011). De los 2,772 clientes modelados, el 39.29% presentó churn, logrando el modelo un ROC-AUC de 0.7255 y un AP de 0.5926. La política óptima seleccionada en validación fue "Contactar a todos" (umbral 0.00), la cual, al aplicarse a los 555 clientes del conjunto de prueba, obtuvo una precisión del 39.28% y un recall del 100%, generando un valor neto simulado de £11,789.08 bajo los supuestos del caso. Se priorizaría el segmento "Champions" por concentrar el mayor valor económico total (£1,035,482), mientras que se descartarían intervenciones costosas en "New_or_Promising" debido a su bajo valor recuperable pese a su alto riesgo. Una limitación crítica es que el CLV utilizado es un proxy de ingresos que ignora márgenes reales y costos de servicio, y la efectividad del 25% es un supuesto no causal. Esta decisión cambiaría si la efectividad real de retención cayera drásticamente; por ejemplo menor al 10%  o si los costos operativos de contacto e incentivo aumentaran, volviendo negativa la ecuación de valor neto.


No repita las respuestas aquí: inclúyalas en un único cierre.


# 12. Informe ejecutivo

In [ ]:
best_threshold = float(best["threshold"])
best_contacted = int(final_result["contacted"])
best_value = float(final_result["net_value"])

print("Resumen automático para apoyar la redacción")
print("-------------------------------------------")
print("Clientes modelados:", len(model_df))
print("Tasa churn:", f"{model_df['churn'].mean():.2%}")
print("ROC-AUC:", f"{roc_auc_score(y_test, proba):.4f}")
print("AP:", f"{average_precision_score(y_test, proba):.4f}")
print("Política elegida en validación:", best["policy"])
print("Clientes evaluados en prueba:", len(test_result))
print("Clientes a contactar:", best_contacted)
print("Valor neto simulado en prueba (£):", f"{best_value:,.2f}")
print("Precision:", f"{final_result['precision']:.2%}")
print("Recall:", f"{final_result['recall']:.2%}")

Resumen automático para apoyar la redacción
-------------------------------------------
Clientes modelados: 2772
Tasa churn: 39.29%
ROC-AUC: 0.7255
AP: 0.5926
Política elegida en validación: Contactar a todos
Clientes evaluados en prueba: 555
Clientes a contactar: 555
Valor neto simulado en prueba (£): 11,789.08
Precision: 39.28%
Recall: 100.00%


**Interpretación:**

El resumen consolidado evidencia el desempeño final del modelo predictivo y la viabilidad de la decisión comercial derivada. Sobre una cohorte de 2,772 clientes con una tasa de abandono del 39.29 %, el modelo exhibe una capacidad discriminatoria moderada (ROC-AUC = 0.7255) y un rendimiento aceptable en la identificación de la clase minoritaria (AP = 0.5926). La evaluación económica determinó que la política óptima es "Contactar a todos" (umbral 0.00), la cual, al aplicarse al conjunto de prueba de 555 clientes, logra un recall del 100 % al capturar la totalidad de los casos reales de churn, aunque con una precisión del 39.28 % debido a la inclusión de falsos positivos. No obstante, esta estrategia genera un valor neto simulado positivo de £11,789.08 en el conjunto de prueba, validando que, bajo los supuestos del caso, el margen recuperado de los clientes retenidos compensa ampliamente los costos operativos de la campaña masiva de retención.

Redacte un cierre de **máximo 8 líneas** con:

1. definición de churn y fecha de corte;
2. número de clientes y tasa de churn;
3. ROC-AUC y AP;
4. política elegida en validación;
5. clientes a contactar en prueba;
6. precision y recall;
7. valor neto simulado en prueba bajo los supuestos;
8. segmento RFM prioritario;
9. una limitación;
10. una condición que haría cambiar la decisión.

### Informe ejecutivo

El churn se define operacionalmente como la inactividad de 90 días posteriores a la fecha de corte (01-sep-2011). De los 2,772 clientes modelados, el 39.29% presentó churn, logrando el modelo un ROC-AUC de 0.7255 y un AP de 0.5926. La política óptima seleccionada en validación fue "Contactar a todos" (umbral 0.00), la cual, al aplicarse a los 555 clientes del conjunto de prueba, obtuvo una precisión del 39.28% y un recall del 100%, generando un valor neto simulado de £11,789.08 bajo los supuestos del caso. Se priorizaría el segmento "Champions" por concentrar el mayor valor económico total, descartando intervenciones costosas en "New_or_Promising" debido a su bajo valor recuperable pese a su alto riesgo. Una limitación crítica es que el CLV proxy ignora márgenes reales y la efectividad del 25% es un supuesto no causal; esta decisión cambiaría si la efectividad real de retención cayera drásticamente o los costos operativos aumentaran, volviendo negativa la ecuación de valor neto.


# 13. Cierre oral
Comparta la decisión de su informe ejecutivo. No se requiere una segunda entrega escrita.

"Recomiendo contactar a todos los clientes del conjunto de prueba porque el valor neto simulado en prueba es £11,789.08 bajo los supuestos del caso; esta decisión cambiaría si la efectividad real de retención cayera drásticamente; por ejemplo, por debajo del 10% o si los costos de contacto e incentivo aumentaran, haciendo que el valor neto esperado sea negativo."


Si la mejor política es no contactar, indíquelo expresamente.


# 14. Evidencia de entrega

Antes de subir a `semana09/`, confirme:

- [ ] Fuente UCI y ID 502 visibles.
- [ ] Limpieza documentada.
- [ ] Target temporal sin leakage.
- [ ] RFM calculado.
- [ ] CLV proxy explicado.
- [ ] Modelo scikit-learn ejecutado.
- [ ] ROC-AUC y AP.
- [ ] Matriz de confusión.
- [ ] Tabla/gráfico de umbrales.
- [ ] Sensibilidad económica.
- [ ] Decisión cuantificada.
- [ ] Limitaciones.
- [ ] Uso de IA declarado.

**Puntaje referencial:** ______ / 20. Este espacio no se utiliza para calificar el laboratorio dirigido.


# Referencias

- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5CG6D.
- Provost, F., & Fawcett, T. *Data Science for Business*.
- Kelleher, J. D., Mac Namee, B., & D'Arcy, A. *Fundamentals of Machine Learning for Predictive Data Analytics*.